# Exercise 2 — Cleaning Café Cozy's Sales Data with pandas

**Goal:** Use pandas to fix a messy sales export — the same kinds of problems you saw in the *Data Cleaning Cycle* slide last week.

**The dataset:** `dirty_sales.csv` — a real-looking export with all the common pain points:
- Missing values (blanks, `N/A`)
- Duplicate rows
- Inconsistent strings (`Latte`, `latte`, ` LATTE `, `Muffin` vs `Blueberry Muffin`)
- Wrong number format (`3,75` instead of `3.75` — German decimal)
- Mixed date formats (`2026-04-01` vs `02/04/2026`)
- Inconsistent country names (`Germany`, `DE`, `deutschland`)

## Step 1 — Load and inspect

`pandas` is the standard Python library for tabular data. A `DataFrame` is like a SQL table or an Excel sheet.

In [ ]:
import pandas as pd

df = pd.read_csv('dirty_sales.csv')
df.head(10)

In [ ]:
# How many rows / columns? What types?
df.info()

In [ ]:
# Missing values per column
df.isnull().sum()

## Step 2 — Remove duplicates

**TODO:** Drop exact duplicate rows. Look up `pandas drop_duplicates`.

In [ ]:
print(f"Before: {len(df)} rows")

# YOUR CODE HERE — assign the de-duplicated DataFrame back to df

print(f"After:  {len(df)} rows")
# Expected: 28 rows after (2 duplicates dropped)

## Step 3 — Standardize product names

The `product` column has trailing spaces and mixed casing. We want one canonical form per product.

**TODO:**
1. Strip whitespace, lowercase, then title-case the values. Use the `.str` accessor: `df['col'].str.strip().str.lower().str.title()`
2. Then replace `'Muffin'` with `'Blueberry Muffin'` so they're treated as one product.

In [ ]:
print("Unique products before:", df['product'].unique())

# YOUR CODE HERE

print("Unique products after: ", df['product'].unique())
# Expected: ['Latte' 'Cappuccino' 'Espresso' 'Blueberry Muffin']

## Step 4 — Fix the price column

Some prices are written `3,75` (German decimal). Some are `N/A`. The column came in as text.

**TODO:**
1. Replace `,` with `.` — first cast to string: `df['price_eur'].astype(str).str.replace(',', '.', regex=False)`
2. Convert to numeric: `pd.to_numeric(..., errors='coerce')` turns unparseable values into `NaN`.

In [ ]:
# YOUR CODE HERE

df['price_eur'].describe()

## Step 5 — Handle missing values

Some `quantity` rows are blank. Some `country` values are missing. Some `price_eur` rows became `NaN` in Step 4.

**Decision time** — these are choices a data analyst makes every day:
- Quantity missing → assume `1`
- Price missing → drop the row (can't compute revenue without it)
- Country missing → fill with `'Unknown'`

**TODO:** apply each rule. Useful: `.fillna(value)`, `.dropna(subset=['col'])`.

In [ ]:
# YOUR CODE HERE

df.isnull().sum()  # should all be zero

## Step 6 — Standardize country names

**TODO:** Map every variant of Germany to `'Germany'`. Lowercase first, then use the `country_map`.

In [ ]:
country_map = {
    'germany':     'Germany',
    'de':          'Germany',
    'deutschland': 'Germany',
    'austria':     'Austria',
    'unknown':     'Unknown',
}

# YOUR CODE HERE

df['country'].value_counts()

## Step 7 — Parse dates

Two formats live in the `date` column. `pd.to_datetime` is forgiving — try `format='mixed'`.

**TODO:** convert the `date` column to real datetimes.

In [ ]:
# YOUR CODE HERE

df.dtypes

## Step 8 — Final check + quick KPI

In [ ]:
df.head()

In [ ]:
# Revenue per product — a preview of next exercise
df['revenue'] = df['price_eur'] * df['quantity']
df.groupby('product')['revenue'].sum().sort_values(ascending=False)

### pandas ↔ SQL cheat sheet

| pandas | SQL |
|---|---|
| `df.drop_duplicates()` | `SELECT DISTINCT` |
| `df['col'].fillna(x)` | `COALESCE(col, x)` |
| `df['col'].str.lower()` | `LOWER(col)` |
| `df.groupby('col').sum()` | `GROUP BY col` |
| `df.dropna(subset=['col'])` | `WHERE col IS NOT NULL` |

Next: combine SQL **and** pandas in one pipeline.